# Token-level方案评估与实验分析（composition-aware）

本Notebook针对你提出的目标：**把整个方案优化成 token-level 处理（数据收集→分析）**。

覆盖两类数据分布：
1. text-only + text-img 混合
2. 仅 text-img

并且优先使用日志中的 token-level 字段：
- `total_supervised_tokens`
- `text_supervised_tokens`
- `image_supervised_tokens`
- `text_token_ratio` / `image_token_ratio`
- `partition_token_ratio`
- `grad_norm_per_token`


In [ ]:
import ast, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='whitegrid', context='talk')


In [ ]:
LOG_PATH = Path('gradient_logs.jsonl')
OUT = Path('analysis_outputs')
CACHE = OUT / 'vector_cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=250000
MAX_VEC_ROWS=60000
MAX_PAIR=30000

assert LOG_PATH.exists(), LOG_PATH
print('using', LOG_PATH.resolve())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try: s=int(v)
    except: return False
    if lo is not None and s<lo: return False
    if hi is not None and s>hi: return False
    if mod is not None and mod>1 and s%mod!=0: return False
    return True

def load_subset(path, usecols=None, partition_in=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not _step_ok(o['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if partition_in is not None and o.get('grad_partition','all') not in partition_in:
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    df=pd.DataFrame(rows)
    if len(df)==0: return df
    for c in ['step','layer_depth','grad_norm','grad_norm_per_token','supervised_token_count','total_supervised_tokens','text_supervised_tokens','image_supervised_tokens','partition_token_ratio','text_token_ratio','image_token_ratio']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
    if 'grad_was_none' in df.columns: df['grad_was_none']=df['grad_was_none'].fillna(False).astype(bool)
    if 'grad_partition' in df.columns: df['grad_partition']=df['grad_partition'].fillna('all')
    return df


## 1) 数据收集质量评估（token-level 完整性）

In [ ]:
meta = load_subset(LOG_PATH, usecols=[
    'step','grad_partition','layer_depth','grad_norm','grad_norm_per_token','grad_was_none',
    'supervised_token_count','total_supervised_tokens','text_supervised_tokens','image_supervised_tokens',
    'partition_token_ratio','text_token_ratio','image_token_ratio',
    'batch_mix_type','batch_text_only_examples','batch_image_examples','batch_video_examples'
], max_rows=MAX_ROWS)

required = ['total_supervised_tokens','text_supervised_tokens','image_supervised_tokens','partition_token_ratio','grad_norm_per_token']
coverage = {k: float(meta[k].notna().mean()) if k in meta.columns and len(meta)>0 else 0.0 for k in required}
summary = {
    'rows_loaded': len(meta),
    'step_min': int(meta.step.min()) if len(meta)>0 else None,
    'step_max': int(meta.step.max()) if len(meta)>0 else None,
    'partitions': sorted(meta.grad_partition.dropna().unique().tolist()) if 'grad_partition' in meta else [],
    'batch_mix_types': sorted(meta.batch_mix_type.dropna().unique().tolist()) if 'batch_mix_type' in meta else [],
    'token_field_coverage': coverage,
}
summary


In [ ]:
# 回退逻辑：如果没有 grad_norm_per_token，用 grad_norm/supervised_token_count 计算
if 'grad_norm_per_token' not in meta.columns or meta['grad_norm_per_token'].notna().mean() == 0:
    if {'grad_norm','supervised_token_count'}.issubset(meta.columns):
        meta['grad_norm_per_token'] = meta['grad_norm'] / meta['supervised_token_count'].clip(lower=1)

# 回退 batch_mix_type
if 'batch_mix_type' not in meta.columns or meta['batch_mix_type'].notna().mean() == 0:
    meta['batch_mix_type'] = 'unknown'

meta[['step','grad_partition','grad_norm_per_token','text_token_ratio','image_token_ratio','batch_mix_type']].head(5)


## 2) 方案级优化建议（可执行）

In [ ]:
advice=[]
if 'text_and_vision' in set(meta['batch_mix_type'].dropna().unique()):
    advice.append('混合数据存在：实验报告中必须单独报告 text_and_vision 子集，避免被纯文本batch稀释。')
if 'vision_only' in set(meta['batch_mix_type'].dropna().unique()):
    advice.append('存在 vision_only：重点解释 token分区冲突（text token vs image token），少做样本级类型冲突叙事。')
if meta['grad_norm_per_token'].notna().mean() < 0.8:
    advice.append('grad_norm_per_token 覆盖率偏低：建议在训练侧固定写入该字段。')
if 'partition_token_ratio' in meta.columns and meta['partition_token_ratio'].notna().mean() >= 0.8:
    advice.append('可按 partition_token_ratio 分箱，比较冲突在高视觉token占比批次是否加剧。')
advice if advice else ['当前token-level字段完整度较好，可直接进行机制分析。']


## 3) Token-level主图（统一）

In [ ]:
plt.figure(figsize=(12,5))
trend = meta.groupby(['step','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
sns.lineplot(data=trend, x='step', y='grad_norm_per_token', hue='grad_partition')
plt.title('Mean grad_norm_per_token over steps')
plt.tight_layout(); plt.savefig(OUT/'fig_token_trend_partition.png', dpi=180); plt.show()

lp = meta.groupby(['layer_depth','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
piv = lp.pivot(index='layer_depth', columns='grad_partition', values='grad_norm_per_token').sort_index()
plt.figure(figsize=(10,8)); sns.heatmap(piv, cmap='magma')
plt.title('Layer x partition (grad_norm_per_token)')
plt.tight_layout(); plt.savefig(OUT/'fig_token_layer_partition_heatmap.png', dpi=200); plt.show()


## 4) 场景A：text-only + text-img 混合

In [ ]:
mixed = meta[meta['batch_mix_type']=='text_and_vision'].copy()
if len(mixed)>0:
    key=['step','layer_depth']
    w=mixed.pivot_table(index=key, columns='grad_partition', values='grad_norm_per_token', aggfunc='mean').reset_index()
    if {'text_only','image_only'}.issubset(w.columns):
        eps=1e-12
        w['dominance']=(w['image_only']-w['text_only'])/(w['image_only']+w['text_only']+eps)
        plt.figure(figsize=(10,4)); sns.histplot(w['dominance'], bins=60, kde=True); plt.axvline(0,color='r',ls='--')
        plt.title('Dominance in mixed batches (token-level)')
        plt.tight_layout(); plt.savefig(OUT/'fig_mixed_dominance_hist.png', dpi=180); plt.show()

        dl=w.groupby('layer_depth',as_index=False)['dominance'].mean()
        plt.figure(figsize=(11,4)); sns.lineplot(data=dl,x='layer_depth',y='dominance',marker='o'); plt.axhline(0,color='r',ls='--')
        plt.title('Layer-wise dominance (mixed batches)')
        plt.tight_layout(); plt.savefig(OUT/'fig_mixed_dominance_layer.png', dpi=180); plt.show()
else:
    print('No text_and_vision rows under current filters.')


## 5) 场景B：仅 text-img（vision-only）

In [ ]:
vision = meta[meta['batch_mix_type'].isin(['vision_only'])].copy()
base = vision if len(vision)>0 else meta.copy()

if len(base)>0:
    if 'grad_was_none' in base.columns:
        nr=base.groupby(['layer_depth','grad_partition'], as_index=False)['grad_was_none'].mean()
        pv=nr.pivot(index='layer_depth', columns='grad_partition', values='grad_was_none').sort_index()
        plt.figure(figsize=(10,8)); sns.heatmap(pv, cmap='Blues')
        plt.title('None-grad ratio (vision-focused)')
        plt.tight_layout(); plt.savefig(OUT/'fig_vision_none_ratio.png', dpi=200); plt.show()

    # token比例与冲突强度关系
    if {'image_token_ratio','grad_partition','grad_norm_per_token'}.issubset(base.columns):
        b=base[base['grad_partition'].isin(['text_only','image_only'])].copy()
        b['image_ratio_bin']=pd.cut(b['image_token_ratio'], bins=[-0.01,0.2,0.4,0.6,0.8,1.0])
        g=b.groupby(['image_ratio_bin','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
        plt.figure(figsize=(12,4)); sns.barplot(data=g, x='image_ratio_bin', y='grad_norm_per_token', hue='grad_partition')
        plt.title('grad_norm_per_token vs image_token_ratio bins')
        plt.tight_layout(); plt.savefig(OUT/'fig_image_ratio_bins_partition.png', dpi=180); plt.show()


## 6) 完整grad向量：方向冲突与容量（SVD）

In [ ]:
def to_npy(g):
    if g is None: return None
    if isinstance(g,(list,tuple,np.ndarray)):
        a=np.asarray(g,dtype=np.float32).reshape(-1)
        h=hashlib.md5((str(a.shape)+str(float(a.sum()))).encode()).hexdigest()
        p=CACHE/f'{h}.npy'
        if not p.exists(): np.save(p,a)
        return str(p)
    if isinstance(g,str):
        s=g.strip()
        if not s: return None
        if s.endswith('.npy') and Path(s).exists(): return s
        for parser in (json.loads, ast.literal_eval):
            try:
                x=parser(s)
                if isinstance(x,(list,tuple)):
                    a=np.asarray(x,dtype=np.float32).reshape(-1)
                    h=hashlib.md5((s[:120]+str(len(a))).encode()).hexdigest()
                    p=CACHE/f'{h}.npy'
                    if not p.exists(): np.save(p,a)
                    return str(p)
            except: pass
    return None

vec = load_subset(LOG_PATH, usecols=['step','param_name','layer_depth','adapter_type','grad_partition','grad','batch_mix_type'], partition_in={'text_only','image_only','all'}, max_rows=MAX_VEC_ROWS)
if 'grad' in vec.columns:
    vec['grad_path']=vec['grad'].apply(to_npy)
    vec=vec[vec['grad_path'].notna()].copy()
else:
    vec=pd.DataFrame()
print('vector rows:', len(vec))


In [ ]:
def cos(a,b,eps=1e-12): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))

if len(vec)>0:
    key=['step','param_name','layer_depth','adapter_type']
    if 'batch_mix_type' in vec.columns: key.append('batch_mix_type')
    t=vec[vec['grad_partition']=='text_only'][key+['grad_path']].rename(columns={'grad_path':'tp'})
    i=vec[vec['grad_partition']=='image_only'][key+['grad_path']].rename(columns={'grad_path':'ip'})
    pair=t.merge(i,on=key,how='inner').head(MAX_PAIR)

    vals=[]
    for _,r in pair.iterrows():
        a=np.load(r['tp']).reshape(-1); b=np.load(r['ip']).reshape(-1)
        m=min(len(a),len(b)); vals.append(np.nan if m==0 else cos(a[:m],b[:m]))
    pair['cos']=vals; pair=pair.dropna(subset=['cos'])

    plt.figure(figsize=(9,4)); sns.histplot(pair['cos'], bins=60, kde=True); plt.axvline(0,color='r',ls='--')
    plt.title('Cos(text,image)'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_text_image.png', dpi=180); plt.show()

    if 'batch_mix_type' in pair.columns:
        plt.figure(figsize=(10,4)); sns.boxplot(data=pair, x='batch_mix_type', y='cos')
        plt.title('Cos(text,image) by composition'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_by_composition.png', dpi=180); plt.show()


In [ ]:
def rank95(X):
    if X.shape[0]<2: return np.nan
    X=X-X.mean(0,keepdims=True)
    s=np.linalg.svd(X,full_matrices=False,compute_uv=False)
    e=np.cumsum(s*s)/(np.sum(s*s)+1e-12)
    return int(np.searchsorted(e,0.95)+1)

if len(vec)>0:
    rec=[]
    gcols=['layer_depth','grad_partition'] + (['batch_mix_type'] if 'batch_mix_type' in vec.columns else [])
    for key,g in vec.groupby(gcols):
        gg=g.head(180)
        if len(gg)<2: continue
        arr=[np.load(p).reshape(-1) for p in gg['grad_path'].tolist()]
        d=min(len(x) for x in arr)
        if d<=1: continue
        X=np.stack([x[:d] for x in arr])
        rec.append((*((key,) if not isinstance(key,tuple) else key), len(gg), d, rank95(X)))

    cols=gcols+['n','dim','rank95']
    rdf=pd.DataFrame(rec,columns=cols)
    if len(rdf)>0:
        rdf.to_csv(OUT/'table_rank95_token_level.csv', index=False, encoding='utf-8-sig')
        plt.figure(figsize=(11,5))
        if 'batch_mix_type' in rdf.columns:
            sns.lineplot(data=rdf, x='layer_depth', y='rank95', hue='grad_partition', style='batch_mix_type', markers=True, dashes=False)
        else:
            sns.lineplot(data=rdf, x='layer_depth', y='rank95', hue='grad_partition', marker='o')
        plt.title('Rank95 by layer/partition/composition')
        plt.tight_layout(); plt.savefig(OUT/'fig_rank95_token_level.png', dpi=180); plt.show()


## 7) 结论模板（token-level）

- 若 `cos(text,image)<0` 层显著集中：说明 token-level 方向冲突成立。  
- 若余弦接近0但 `rank95` 上升：说明共享低秩空间容量竞争更关键。  
- 混合数据时应报告 `text_and_vision` 子集结果；仅 text-img 时应把冲突定义为 token 分区冲突而非样本类型冲突。  
- 建议第4章方法直接作用于“低余弦 + 高rank95”的关键层。
